# Exploratory analysis with SQL
Loads the course launch table into an in-memory SQLite database and answers the lab questions with SQL (`sqlite3` + `pandas.read_sql_query`).

In [1]:
import sqlite3

import pandas as pd

con = sqlite3.connect(':memory:')
raw = pd.read_csv('../data/raw/Spacex.csv')
raw.to_sql('SPACEXTBL', con, index=False)
con.execute('CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL')


def sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, con)


sql('SELECT COUNT(*) AS launches FROM SPACEXTABLE')

,launches
0,101


### Display the names of the unique launch sites in the space mission

In [2]:
sql("""
SELECT DISTINCT "Launch_Site"
FROM SPACEXTABLE
""")

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Display 5 records where launch sites begin with the string 'CCA'

In [3]:
sql("SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Display the total payload mass carried by boosters launched by NASA (CRS)

In [4]:
sql("""
SELECT SUM("PAYLOAD_MASS__KG_") AS Total_Payload_Mass
FROM SPACEXTABLE
WHERE "Customer" = 'NASA (CRS)'
""")

,Total_Payload_Mass
0,45596


### Display average payload mass carried by booster version F9 v1.1

In [5]:
sql("""
SELECT AVG("PAYLOAD_MASS__KG_") AS Average_Payload_Mass
FROM SPACEXTABLE
WHERE "Booster_Version" = 'F9 v1.1'
""")

,Average_Payload_Mass
0,2928.4


### List the date when the first successful landing outcome in ground pad was achieved

In [6]:
sql("""
SELECT MIN("Date") AS First_Successful_Landing_Date
FROM SPACEXTABLE
WHERE TRIM("Mission_Outcome") = 'Success' AND "Landing_Outcome" = 'Success (ground pad)'
""")

,First_Successful_Landing_Date
0,2015-12-22


### List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000

In [7]:
sql("""
SELECT DISTINCT "Booster_Version"
FROM SPACEXTABLE
WHERE TRIM("Mission_Outcome") = 'Success' 
  AND "Landing_Outcome" = 'Success (drone ship)'
  AND "PAYLOAD_MASS__KG_" > 4000 
  AND "PAYLOAD_MASS__KG_" < 6000
""")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### List the total number of successful and failure mission outcomes

In [8]:
sql("""
SELECT TRIM("Mission_Outcome") AS Mission_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
GROUP BY TRIM("Mission_Outcome")
""")

,Mission_Outcome,Outcome_Count
0,Failure (in flight),1
1,Success,99
2,Success (payload status unclear),1


### List the names of the booster_versions which have carried the maximum payload mass. Use a subquery

In [9]:
sql("""
SELECT "Booster_Version"
FROM SPACEXTABLE
WHERE "PAYLOAD_MASS__KG_" = (
    SELECT MAX("PAYLOAD_MASS__KG_")
    FROM SPACEXTABLE
)
""")

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### List the records which will display the month names, failure landing_outcomes in drone ship, booster versions, launch_site for the months in year 2015.

In [10]:
sql("""
SELECT 
    CASE 
        WHEN SUBSTR("Date", 6, 2) = '01' THEN 'January'
        WHEN SUBSTR("Date", 6, 2) = '02' THEN 'February'
        WHEN SUBSTR("Date", 6, 2) = '03' THEN 'March'
        WHEN SUBSTR("Date", 6, 2) = '04' THEN 'April'
        WHEN SUBSTR("Date", 6, 2) = '05' THEN 'May'
        WHEN SUBSTR("Date", 6, 2) = '06' THEN 'June'
        WHEN SUBSTR("Date", 6, 2) = '07' THEN 'July'
        WHEN SUBSTR("Date", 6, 2) = '08' THEN 'August'
        WHEN SUBSTR("Date", 6, 2) = '09' THEN 'September'
        WHEN SUBSTR("Date", 6, 2) = '10' THEN 'October'
        WHEN SUBSTR("Date", 6, 2) = '11' THEN 'November'
        WHEN SUBSTR("Date", 6, 2) = '12' THEN 'December'
    END AS "Month_Name",
    "Landing_Outcome",
    "Booster_Version",
    "Launch_Site"
FROM SPACEXTABLE
WHERE "Landing_Outcome" = 'Failure (drone ship)'
  AND SUBSTR("Date", 1, 4) = '2015'
""")

,Month_Name,Landing_Outcome,Booster_Version,Launch_Site
0,January,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,April,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [11]:
sql("""
SELECT "Landing_Outcome", COUNT(*) AS Outcome_Count
FROM SPACEXTABLE
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing_Outcome"
ORDER BY Outcome_Count DESC
""")

,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
